# Notebook 1 (Local, RTX 4060 optimized) — Phase A: Data Building

Optimized for NVIDIA RTX 4060 8 GB VRAM.

**Key optimizations applied:**
- `num_return_sequences=K` — all 4 continuations in one forward pass (~3× faster than sequential)
- Flash Attention 2 — faster attention kernel (Ada Lovelace / Ampere+)
- TF32 matmul — free ~10% throughput on RTX 30/40 series
- `torch.compile` — JIT fusion after ~30 s warmup
- `MAX_NEW=2048` restored (enough VRAM at 3 GB model + 5 GB KV cache headroom)

**Expected speed:** ~5-15 s per question (K=4 in one shot) → full 966 rows in ~2-4 hours.

In [1]:
# Cell 1 — Install Dependencies
# Run once, then restart the kernel.
import subprocess, sys

pkgs = [
    "transformers", "accelerate", "tqdm", "datasets",
    "math-verify", "sympy", "huggingface_hub",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Flash Attention 2 — pre-built wheel for CUDA 12.x + PyTorch 2.x
# If this fails, the notebook falls back to standard attention automatically.
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "flash-attn", "--no-build-isolation",
    ])
    print("flash-attn installed.")
except Exception:
    print("[warn] flash-attn install failed — will use standard attention (still fast on RTX 4060).")

[warn] flash-attn install failed — will use standard attention (still fast on RTX 4060).


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [29 lines of output]
      /home/thach/miniconda3/envs/P-ALIGN/lib/python3.10/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
        warn(
      
      
      torch.__version__  = 2.8.0+cu128
      
      
      <string>:106: UserWarning: flash_attn was requested, but nvcc was not found.  Are you sure your environment has nvcc available?  If you're installing within a container from https://hub.docker.com/r/pytorch/pytorch, only images whose names contain 'devel' will provide nvcc.
      Traceback (most recent call last):
        File "/home/thach/miniconda3/envs/P-ALIGN/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_p

In [2]:
# Cell 2 — Config
import os

# ── CHANGE THESE ────────────────────────────────────────────────────────────
BASE_DIR     = os.path.join(os.path.expanduser("~"), "P-ALIGN-data")
MODEL_DIR    = os.path.join(BASE_DIR, "models", "Qwen2.5-1.5B-Instruct")
SUBSET_SIZE  = None    # set to e.g. 50 for a quick end-to-end test; None = all 966
USE_COMPILE  = True    # torch.compile — adds ~30 s warmup, then faster; set False to skip
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(os.path.join(BASE_DIR, "data"),   exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)
print(f"Output directory : {BASE_DIR}")
print(f"torch.compile    : {USE_COMPILE}")

Output directory : /home/thach/P-ALIGN-data
torch.compile    : True


In [3]:
# Cell 3 — GPU Setup
import torch

assert torch.cuda.is_available(), "CUDA not found — check your driver / PyTorch install."

DEVICE      = "cuda"
TORCH_DTYPE = torch.float16

# TF32: free ~10% throughput on Ampere/Ada with no accuracy loss
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"dtype: {TORCH_DTYPE}  |  TF32: ON")

GPU  : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM : 8.2 GB
dtype: torch.float16  |  TF32: ON


In [4]:
# Cell 4 — Download Model (skip if already downloaded)
from huggingface_hub import snapshot_download

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("Downloading Qwen2.5-1.5B-Instruct (~3 GB) ...")
    snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=MODEL_DIR)
    print(f"Downloaded → {MODEL_DIR}")
else:
    print(f"[Resume] Model already at {MODEL_DIR}")

[Resume] Model already at /home/thach/P-ALIGN-data/models/Qwen2.5-1.5B-Instruct


/home/thach/miniconda3/envs/P-ALIGN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Cell 5 — Load & Join Datasets → prefixes.jsonl
import re, json, unicodedata
from datasets import load_dataset

PREFIX_FILE = os.path.join(BASE_DIR, "data", "prefixes.jsonl")

if os.path.exists(PREFIX_FILE):
    with open(PREFIX_FILE) as f:
        rows = [json.loads(l) for l in f]
    print(f"[Resume] Loaded {len(rows)} rows from {PREFIX_FILE}")
else:
    # P-ALIGN Alpaca format:
    #   instruction = fixed system prompt (ignore)
    #   input       = math question        ← join key
    #   output      = prefix with <Begin_of_Prefix> tag
    palign = load_dataset("qizheyanger/P-ALIGN", split="train")
    print(f"P-ALIGN: {len(palign)} rows, columns: {palign.column_names}")

    _PREFIX_TAG = re.compile(r"<Begin_of_Prefix>\s*", re.IGNORECASE)
    palign_parsed = [
        {"question": item["input"].strip(),
         "prefix":   _PREFIX_TAG.sub("", item["output"]).strip()}
        for item in palign
    ]

    s1k = load_dataset("simplescaling/s1K-1.1", split="train")
    print(f"s1K-1.1: {len(s1k)} rows")

    def _norm(text: str) -> str:
        return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text).lower().strip())

    s1k_lookup      = {item["question"].strip(): {"full_cot": item["deepseek_thinking_trajectory"],
                                                   "answer":   str(item["solution"]).strip()}
                       for item in s1k}
    s1k_lookup_norm = {_norm(k): v for k, v in s1k_lookup.items()}

    rows, n_miss = [], 0
    for i, parsed in enumerate(palign_parsed):
        q      = parsed["question"]
        lookup = s1k_lookup.get(q) or s1k_lookup_norm.get(_norm(q))
        if lookup is None:
            q_fp   = _norm(q)[:120]
            lookup = next((v for k, v in s1k_lookup_norm.items() if k[:120] == q_fp), None)
        if lookup is None:
            n_miss += 1; continue
        rows.append({"id": i, "question": q, "prefix": parsed["prefix"],
                     "answer": lookup["answer"], "full_cot": lookup["full_cot"]})

    print(f"Joined {len(rows)} samples ({n_miss} unmatched)")
    with open(PREFIX_FILE, "w") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved → {PREFIX_FILE}")

[Resume] Loaded 966 rows from /home/thach/P-ALIGN-data/data/prefixes.jsonl


In [6]:
# Cell 6 — Load Student Model
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# Try Flash Attention 2 first; fall back gracefully if not installed
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        dtype=TORCH_DTYPE,
        device_map=DEVICE,
        attn_implementation="flash_attention_2",
    )
    print("Attention backend : Flash Attention 2")
except Exception as e:
    print(f"Flash Attention 2 unavailable ({e}) — using standard SDPA.")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        dtype=TORCH_DTYPE,
        device_map=DEVICE,
    )

model.eval()
model.config.use_cache = True   # KV cache ON for inference

# torch.compile — fuses ops, ~10-20% extra speedup after warmup
if USE_COMPILE:
    model = torch.compile(model, mode="reduce-overhead")
    print("torch.compile    : ON (reduce-overhead) — first call takes ~30 s to compile.")

used_gb = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used by model: {used_gb:.1f} GB  |  free: {vram_gb - used_gb:.1f} GB")

Flash Attention 2 unavailable (FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.) — using standard SDPA.
torch.compile    : ON (reduce-overhead) — first call takes ~30 s to compile.
VRAM used by model: 3.1 GB  |  free: 5.1 GB


In [ ]:
# Cell 7 — Hyperparameters
K          = 4      # continuations per question — generated in ONE forward pass
TEMP       = 0.8
TOP_P      = 0.95
MAX_NEW    = 1024   # full 2048 — fits easily in 8 GB with 3 GB model weight
TAU_LOW    = 0.2
T_MAX      = 3
LAMBDA_LEN = 0.0
EPSILON    = 1e-6
CLIP_C     = 3.0
print(f"K={K}  MAX_NEW={MAX_NEW}  TEMP={TEMP}")

K=4  MAX_NEW=2048  TEMP=0.8


In [8]:
# Cell 8 — Helper Functions
import re, math, signal
from math_verify import parse, verify

def build_prompt(question: str, prefix: str) -> str:
    return (
        "Please continue from the draft and solve the problem step by step, "
        "and put your final answer within \\boxed{}. "
        "I will provide you with some prior knowledge as a draft to assist you.\n"
        f"*Question*: {question}\n"
        f"*Prefix*: {prefix}"
    )

def sample_k_continuations(question: str, prefix: str, k: int = K) -> list:
    """Generate all K continuations in ONE forward pass using num_return_sequences."""
    prompt = build_prompt(question, prefix)
    text   = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW,
            do_sample=True,
            temperature=TEMP,
            top_p=TOP_P,
            num_return_sequences=k,       # ← K outputs in one shot
            pad_token_id=tokenizer.eos_token_id,
        )
    # out shape: [K, prompt_len + gen_len]
    prompt_len = inputs.input_ids.shape[1]
    conts = [
        tokenizer.decode(out[i][prompt_len:], skip_special_tokens=True)
        for i in range(k)
    ]

    answers = [extract_boxed(c) for c in conts]
    if len(set(answers)) == 1:
        print("  [warn] All K continuations identical.")
    return conts

def extract_boxed(text: str) -> str:
    m = re.findall(r"\\boxed\{([^}]*)\}", text)
    return m[-1] if m else ""

def verify_answer(pred: str, gold: str, timeout_sec: int = 5) -> int:
    def _h(s, f): raise TimeoutError
    if hasattr(signal, "SIGALRM"):
        old = signal.signal(signal.SIGALRM, _h)
        signal.alarm(timeout_sec)
    try:
        ans = extract_boxed(pred)
        if not ans: return 0
        return int(bool(verify(parse(ans), parse("$" + str(gold) + "$"))))
    except Exception:
        return 0
    finally:
        if hasattr(signal, "SIGALRM"):
            signal.alarm(0); signal.signal(signal.SIGALRM, old)

def compute_rewards(conts, answer):
    return [float(verify_answer(y, answer)) - LAMBDA_LEN * len(y.split()) for y in conts]

def compute_advantage(rewards):
    mu    = sum(rewards) / len(rewards)
    sigma = math.sqrt(sum((r - mu)**2 for r in rewards) / len(rewards)) + EPSILON
    return [(r - mu) / sigma for r in rewards]

def classify_group(rewards):
    if all(r > 0 for r in rewards):  return "all_correct"
    if all(r <= 0 for r in rewards): return "all_wrong"
    return "mixed"

def sentence_split(text):
    return [s.strip() + ("" if s.strip().endswith(".") else ".")
            for s in text.split(". ") if s.strip()]

def build_prefix_ladder(row):
    sents = sentence_split(row.get("full_cot", ""))
    return [" ".join(sents[:i+1]) for i in range(len(sents))]

def get_next_prefix(current, ladder):
    return next((c for c in ladder if len(c) > len(current)), None)

print("Helpers defined.")

Helpers defined.


In [9]:
# Cell 9 — Main Data-Building Loop
from tqdm import tqdm

SFT_L1_FILE = os.path.join(BASE_DIR, "data", "sft_data_L1.jsonl")
SFT_L2_FILE = os.path.join(BASE_DIR, "data", "sft_data_L2.jsonl")
DPO_FILE    = os.path.join(BASE_DIR, "data", "dpo_pairs.jsonl")
STATS_FILE  = os.path.join(BASE_DIR, "data", "build_stats.json")

# Resume: skip already-processed source IDs
processed_ids = set()
for fpath in [SFT_L1_FILE, SFT_L2_FILE]:
    if os.path.exists(fpath):
        with open(fpath) as f:
            for line in f:
                processed_ids.add(json.loads(line).get("source_id", ""))
print(f"Already processed: {len(processed_ids)} source IDs")

with open(PREFIX_FILE) as f:
    all_rows = [json.loads(l) for l in f]

# Optional subset for testing
run_rows = all_rows[:SUBSET_SIZE] if SUBSET_SIZE else all_rows
print(f"Processing {len(run_rows)} rows (SUBSET_SIZE={SUBSET_SIZE})")

stats = {"total": 0, "all_correct": 0, "all_wrong": 0, "mixed": 0, "feedback_triggered": 0}

f_l1  = open(SFT_L1_FILE, "a", encoding="utf-8")
f_l2  = open(SFT_L2_FILE, "a", encoding="utf-8")
f_dpo = open(DPO_FILE,    "a", encoding="utf-8")

try:
    for row in tqdm(run_rows, desc="Building dataset"):
        src_id = str(row.get("id", row["question"][:40]))
        if src_id in processed_ids:
            continue

        question = row["question"]
        prefix   = row["prefix"]
        answer   = str(row["answer"])
        ladder   = build_prefix_ladder(row)
        stats["total"] += 1

        for _ in range(T_MAX):
            conts     = sample_k_continuations(question, prefix, K)
            rewards   = compute_rewards(conts, answer)
            pass_rate = sum(1 for r in rewards if r > 0) / K
            if pass_rate < TAU_LOW:
                next_p = get_next_prefix(prefix, ladder)
                if next_p:
                    prefix = next_p
                    stats["feedback_triggered"] += 1
                    continue
            break

        group_type = classify_group(rewards)
        stats[group_type] = stats.get(group_type, 0) + 1

        if group_type == "all_wrong":
            continue

        advantages = compute_advantage(rewards)
        mu_r       = sum(rewards) / len(rewards)
        sum_w_l1   = sum(1.0 for r in rewards if r >= mu_r) + EPSILON
        sum_w_l2   = sum(max(a, 0) for a in advantages) + EPSILON

        for k_i, (y, r, a_k) in enumerate(zip(conts, rewards, advantages)):
            full_seq = prefix + "\n" + y
            w_l1 = (1.0 / sum_w_l1) if r >= mu_r else 0.0
            w_l2 = min(max(a_k, 0), CLIP_C) / sum_w_l2
            base = {"source_id": src_id, "question": question, "prefix": prefix,
                    "continuation": y, "full_sequence": full_seq,
                    "answer": answer, "reward": r, "pass_rate": pass_rate}
            if w_l1 > 0:
                f_l1.write(json.dumps({**base, "weight": w_l1}, ensure_ascii=False) + "\n")
            if w_l2 > 0:
                f_l2.write(json.dumps({**base, "weight": w_l2}, ensure_ascii=False) + "\n")

        if group_type == "mixed":
            best_k  = max(range(K), key=lambda k: rewards[k])
            worst_k = min(range(K), key=lambda k: rewards[k])
            if rewards[best_k] > rewards[worst_k]:
                f_dpo.write(json.dumps({
                    "source_id": src_id, "question": question, "prefix": prefix,
                    "chosen": conts[best_k], "rejected": conts[worst_k],
                    "reward_chosen": rewards[best_k], "reward_rejected": rewards[worst_k],
                }, ensure_ascii=False) + "\n")

        # Flush every row so progress survives a crash
        f_l1.flush(); f_l2.flush(); f_dpo.flush()

finally:
    f_l1.close(); f_l2.close(); f_dpo.close()

with open(STATS_FILE, "w") as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))

Already processed: 2 source IDs
Processing 966 rows (SUBSET_SIZE=None)


Building dataset:   1%|          | 5/966 [04:12<14:57:32, 56.04s/it]

  [warn] All K continuations identical.


Building dataset:   1%|          | 5/966 [04:36<14:45:38, 55.30s/it]


KeyboardInterrupt: 

In [ ]:
# Cell 10 — Sanity Check
for label, fpath in [("L1", SFT_L1_FILE), ("L2", SFT_L2_FILE), ("DPO", DPO_FILE)]:
    n = sum(1 for _ in open(fpath)) if os.path.exists(fpath) else 0
    print(f"{label}: {n} samples")

if os.path.exists(STATS_FILE):
    print(json.load(open(STATS_FILE)))